# Credit Risk Scoring Engine — Model Evaluation, Tuning & Thresholding

## Overview & Business Objectives
In credit risk management, evaluating classification performance requires looking beyond simple Accuracy. Approving a defaulting loan (False Negative) carries a financial loss up to 5x greater than turning away a creditworthy applicant (False Positive).

This notebook completes four critical analytical goals:
1. **Comprehensive Baseline Model Evaluation**: Evaluation across Accuracy, Precision, Recall, F1-Score, ROC-AUC, PR-AUC, Brier Score calibration, ROC curves, Precision-Recall curves, and Confusion Matrices.
2. **Hyperparameter Optimization (HPO)**: Systematic tuning via 5-fold `StratifiedKFold` cross-validation on `X_train` for Logistic Regression, Random Forest, and XGBoost.
3. **Decision Threshold Optimization**: Analysis across cutoff thresholds from 0.10 to 0.70 to minimize overall credit portfolio default losses.
4. **Model Selection Justification**: Final candidate comparison table and selection rationale.

In [2]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, precision_recall_curve, auc, brier_score_loss,
    confusion_matrix, classification_report, roc_curve
)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Load processed data
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').values.ravel()
y_test = pd.read_csv('../data/processed/y_test.csv').values.ravel()

# Load baseline models
baseline_models = joblib.load('../models/baseline_models.joblib')
print(f"Loaded {len(baseline_models)} baseline models for evaluation.")


In [3]:
baseline_results = []
model_probas = {}
model_preds = {}

for name, model in baseline_models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    model_preds[name] = y_pred
    model_probas[name] = y_proba
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_proba)
    
    p_prec, p_rec, _ = precision_recall_curve(y_test, y_proba)
    pr_auc = auc(p_rec, p_prec)
    brier = brier_score_loss(y_test, y_proba)
    
    baseline_results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall (Bad)': round(rec, 4),
        'F1-Score': round(f1, 4),
        'ROC-AUC': round(roc_auc, 4),
        'PR-AUC': round(pr_auc, 4),
        'Brier Score': round(brier, 4)
    })

df_baseline_eval = pd.DataFrame(baseline_results)
print("=== Baseline Models Test Evaluation Table ===")
print(df_baseline_eval.to_string(index=False))


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot ROC Curves
for name, y_proba in model_probas.items():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_val = roc_auc_score(y_test, y_proba)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC = {auc_val:.3f})")

axes[0].plot([0, 1], [0, 1], 'k--', label='Random Chance (AUC = 0.500)')
axes[0].set_title('Receiver Operating Characteristic (ROC) Curves', fontsize=12)
axes[0].set_xlabel('False Positive Rate (1 - Specificity)')
axes[0].set_ylabel('True Positive Rate (Recall)')
axes[0].legend(fontsize=8, loc='lower right')
axes[0].grid(True, alpha=0.3)

# Plot PR Curves
for name, y_proba in model_probas.items():
    p_prec, p_rec, _ = precision_recall_curve(y_test, y_proba)
    pr_auc_val = auc(p_rec, p_prec)
    axes[1].plot(p_rec, p_prec, label=f"{name} (PR-AUC = {pr_auc_val:.3f})")

axes[1].axhline(y=0.30, color='k', linestyle='--', label='Baseline Default Rate (0.30)')
axes[1].set_title('Precision-Recall (PR) Curves', fontsize=12)
axes[1].set_xlabel('Recall (Bad-Credit Capture)')
axes[1].set_ylabel('Precision (Positive Predictive Value)')
axes[1].legend(fontsize=8, loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [5]:
fig, ax = plt.subplots(figsize=(8, 6))

for name in ['Logistic Regression', 'Logistic Regression (Balanced)', 'Random Forest', 'Gradient Boosting', 'XGBoost']:
    prob_true, prob_pred = calibration_curve(y_test, model_probas[name], n_bins=5)
    ax.plot(prob_pred, prob_true, marker='o', label=name)

ax.plot([0, 1], [0, 1], 'k--', label='Perfectly Calibrated')
ax.set_title('Probability Calibration Curves (Reliability Diagram)', fontsize=12)
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives (Actual Default Rate)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [6]:
print("=== Starting Hyperparameter Optimization (5-Fold Stratified CV) ===")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 1. Tune Random Forest
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid={
        'n_estimators': [100, 200],
        'max_depth': [5, 8],
        'class_weight': [None, 'balanced']
    },
    cv=cv,
    scoring='roc_auc'
)
rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_

# 2. Tune XGBoost
xgb_grid = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    param_grid={
        'n_estimators': [100],
        'max_depth': [3, 4],
        'learning_rate': [0.05, 0.1]
    },
    cv=cv,
    scoring='roc_auc'
)
xgb_grid.fit(X_train, y_train)
best_xgb = xgb_grid.best_estimator_

# 3. Tune Logistic Regression
lr_grid = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=1000),
    param_grid={
        'C': [0.1, 0.5, 1.0, 5.0],
        'class_weight': [None, 'balanced']
    },
    cv=cv,
    scoring='roc_auc'
)
lr_grid.fit(X_train, y_train)
best_lr = lr_grid.best_estimator_

print(f"Tuned Random Forest Best CV ROC-AUC:    {rf_grid.best_score_:.4f} | Params: {rf_grid.best_params_}")
print(f"Tuned XGBoost Best CV ROC-AUC:          {xgb_grid.best_score_:.4f} | Params: {xgb_grid.best_params_}")
print(f"Tuned Logistic Regression Best CV ROC: {lr_grid.best_score_:.4f} | Params: {lr_grid.best_params_}")


In [7]:
tuned_models = {
    'Tuned Logistic Regression': best_lr,
    'Tuned Random Forest': best_rf,
    'Tuned XGBoost': best_xgb
}

tuned_results = []
tuned_probas = {}

for name, model in tuned_models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    tuned_probas[name] = y_proba
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_proba)
    
    p_prec, p_rec, _ = precision_recall_curve(y_test, y_proba)
    pr_auc = auc(p_rec, p_prec)
    brier = brier_score_loss(y_test, y_proba)
    
    tuned_results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall (Bad)': round(rec, 4),
        'F1-Score': round(f1, 4),
        'ROC-AUC': round(roc_auc, 4),
        'PR-AUC': round(pr_auc, 4),
        'Brier Score': round(brier, 4)
    })

df_tuned_eval = pd.DataFrame(tuned_results)
print("=== Tuned Models Test Evaluation Table ===")
print(df_tuned_eval.to_string(index=False))


In [8]:
target_model_name = 'Tuned Logistic Regression'
target_proba = tuned_probas[target_model_name]

thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70]
thresh_eval = []

for t in thresholds:
    pred_t = (target_proba >= t).astype(int)
    cm = confusion_matrix(y_test, pred_t)
    tn, fp, fn, tp = cm.ravel()
    
    acc_t = accuracy_score(y_test, pred_t)
    prec_t = precision_score(y_test, pred_t, zero_division=0)
    rec_t = recall_score(y_test, pred_t, zero_division=0)
    f1_t = f1_score(y_test, pred_t, zero_division=0)
    
    # Financial Risk Index: Loss = 5 * FN + 1 * FP
    cost_index = 5 * fn + 1 * fp
    
    thresh_eval.append({
        'Threshold': t,
        'TP (Bad Caught)': tp,
        'FP (Good Rejected)': fp,
        'FN (Bad Approved)': fn,
        'TN (Good Approved)': tn,
        'Accuracy': round(acc_t, 4),
        'Precision': round(prec_t, 4),
        'Recall (Bad Capture)': round(rec_t, 4),
        'F1-Score': round(f1_t, 4),
        'Cost Index (5*FN+FP)': cost_index
    })

df_thresh = pd.DataFrame(thresh_eval)
print(f"=== Threshold Optimization Analysis ({target_model_name}) ===")
print(df_thresh.to_string(index=False))

optimal_row = df_thresh.loc[df_thresh['Cost Index (5*FN+FP)'].idxmin()]
print(f"\nOptimal Threshold by Financial Loss: {optimal_row['Threshold']:.2f} (Recall: {optimal_row['Recall (Bad Capture)']*100:.1f}%, F1: {optimal_row['F1-Score']:.4f})")


In [9]:
os.makedirs('../models', exist_ok=True)
joblib.dump(tuned_models, '../models/tuned_models.joblib')

threshold_config = {
    'selected_model': target_model_name,
    'optimal_threshold': float(optimal_row['Threshold']),
    'baseline_threshold': 0.50,
    'recall_at_optimal': float(optimal_row['Recall (Bad Capture)']),
    'precision_at_optimal': float(optimal_row['Precision']),
    'f1_at_optimal': float(optimal_row['F1-Score']),
    'cost_index_optimal': int(optimal_row['Cost Index (5*FN+FP)'])
}
with open('../models/threshold_config.json', 'w') as f:
    json.dump(threshold_config, f, indent=2)

print("Saved tuned models to ../models/tuned_models.joblib")
print("Saved threshold config to ../models/threshold_config.json")
